# Experiment 3 — Strategic deconfliction between two UAVs

This experiment evaluates strategic deconfliction when two proposed UAV operations would reach the same point at the same time. The same scenario is executed without strategic deconfliction and with the built-in recursive time-shift algorithm.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from musas.airspace import Airspace
from musas.algorithms import (
    FirstOrderDynamics,
    MinimumSeparationConflictPrediction,
    RecursiveTimeShiftDeconfliction,
    WaypointGuidance,
)
from musas.domain import FlightPlan, Waypoint
from musas.simulation import (
    Simulation,
    SimulationDataset,
    VisualizationConfig,
    animate_dataset,
    calculate_metrics,
    plot_dataset,
)

OUTPUT_DIRECTORY = Path("outputs") / "output_exp3"
DATASET_DIRECTORY = OUTPUT_DIRECTORY / "datasets"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
DATASET_DIRECTORY.mkdir(exist_ok=True)

## Scenario configuration

UAV A flies west-to-east and UAV B flies south-to-north. Their original plans cross the origin at `t = 17 s`, at the same altitude. UAV A is submitted first so that its plan is already approved when UAV B is evaluated.

In [ ]:
SIMULATION_END = 50.0
ORIGINAL_ENCOUNTER_TIME = 17.0
MINIMUM_SEPARATION_DISTANCE = 30.0
COLLISION_DISTANCE = 5.0
TIME_SHIFT_STEP = 5.0

def create_flight_plans():
    return {
        "UAV A": FlightPlan([
            Waypoint(-100.0, 0.0, 100.0, 2.0),
            Waypoint(100.0, 0.0, 100.0, 32.0),
        ]),
        "UAV B": FlightPlan([
            Waypoint(0.0, -100.0, 100.0, 2.0),
            Waypoint(0.0, 100.0, 100.0, 32.0),
        ]),
    }

## Run the baseline and strategic-deconfliction cases

In the SD case, the conflict predictor evaluates the four-dimensional plans and `RecursiveTimeShiftDeconfliction` delays UAV B until the candidate plan is conflict-free.

In [ ]:
CASES = {
    "without_sd": "Without strategic deconfliction",
    "with_sd": "With strategic deconfliction",
}

def run_case(case_name):
    dataset_path = DATASET_DIRECTORY / f"{case_name}.sqlite"
    simulation_arguments = {}

    if case_name == "with_sd":
        conflict_prediction = MinimumSeparationConflictPrediction(
            minimum_separation_distance=MINIMUM_SEPARATION_DISTANCE,
            sampling_interval=0.25,
        )
        deconfliction = RecursiveTimeShiftDeconfliction(
            conflict_prediction_algorithm=conflict_prediction,
            time_shift_step=TIME_SHIFT_STEP,
            maximum_time_shifts=10,
        )
        simulation_arguments = {
            "strategic_conflict_prediction_algorithm": conflict_prediction,
            "strategic_deconfliction_algorithm": deconfliction,
        }

    simulation = Simulation(
        airspace=Airspace(),
        flight_update_interval=0.1,
        trajectory_record_interval=0.1,
        message_types_to_log=(),
        dataset_path=dataset_path,
        dataset_metadata={
            "experiment": "strategic_deconfliction_between_uavs",
            "case": case_name,
        },
        overwrite_dataset=True,
        **simulation_arguments,
    )

    for uav_name, flight_plan in create_flight_plans().items():
        simulation.add_uav(
            name=uav_name,
            flight_plan=flight_plan,
            dynamics_algorithm=FirstOrderDynamics(
                max_acceleration=100.0,
                velocity_time_constant=0.0,
            ),
            guidance_algorithm=WaypointGuidance(
                min_speed=0.0,
                max_speed=15.0,
            ),
            submission_time=0.0 if uav_name == "UAV A" else 0.5,
            waypoint_acceptance_radius=0.5,
        )

    simulation.run(until=SIMULATION_END)
    simulation.close()
    return dataset_path

dataset_paths = {case_name: run_case(case_name) for case_name in CASES}
dataset_paths

## Metrics

A successful strategic resolution should modify one flight, eliminate the loss-of-separation and collision events, and maintain at least the required separation distance.

In [ ]:
print(
    f"{'Case':<38} {'conflicts':>10} {'modified':>10} "
    f"{'LOS':>6} {'collisions':>11} {'minimum sep.':>14} "
    f"{'updates A':>10} {'updates B':>10}"
)
for case_name, dataset_path in dataset_paths.items():
    with SimulationDataset(dataset_path) as dataset:
        metrics = calculate_metrics(
            dataset,
            minimum_separation_distance=MINIMUM_SEPARATION_DISTANCE,
            collision_distance=COLLISION_DISTANCE,
        )
    minimum_separation = metrics["minimum_pairwise_separation"]
    minimum_separation_text = (
        "n/a"
        if minimum_separation is None
        else f"{minimum_separation:.2f}"
    )
    print(
        f"{CASES[case_name]:<38} "
        f"{metrics['detected_strategic_conflicts']:10d} "
        f"{metrics['modified_flights']:10d} "
        f"{metrics['loss_of_separation_events']:6d} "
        f"{metrics['collision_events']:11d} "
        f"{minimum_separation_text:>14} "
        f"{metrics['per_uav']['UAV A']['updates']:10d} "
        f"{metrics['per_uav']['UAV B']['updates']:10d}"
    )

## Authorized departure and arrival times

The spatial path remains unchanged. The strategic counterproposal resolves the conflict by shifting all timestamps of UAV B.

In [ ]:
for case_name, dataset_path in dataset_paths.items():
    print(f"\n{CASES[case_name]}:")
    with SimulationDataset(dataset_path) as dataset:
        for uav_name in ("UAV A", "UAV B"):
            versions = dataset.get_flight_plan_versions(uav_name)
            plan = dataset.get_flight_plan(
                uav_name,
                "authorized",
                versions[-1],
            )
            print(
                f"  {uav_name}: departure={plan.waypoint_list[0].t:.1f} s, "
                f"arrival={plan.waypoint_list[-1].t:.1f} s"
            )

## Encounter-time figures

The figures use `plot_dataset()` at the original encounter time. Without SD, both UAVs occupy the intersection. With SD, UAV B is still far from it.

In [ ]:
def visualization_config(case_name):
    return VisualizationConfig(
        uav_names=("UAV A", "UAV B"),
        view="3d",
        show_real_trajectories=True,
        show_current_positions=True,
        show_flight_plans=True,
        show_sensor_reports=False,
        show_tracks=False,
        show_geofences=False,
        show_wind_zones=False,
        show_obstacles=False,
        frame_step=0.5,
        figure_size=(8.0, 8.0),
        title=CASES[case_name],
    )

for case_name, dataset_path in dataset_paths.items():
    figure_path = OUTPUT_DIRECTORY / f"{case_name}.png"
    figure = plot_dataset(
        dataset_path=dataset_path,
        config=visualization_config(case_name),
        timestamp=ORIGINAL_ENCOUNTER_TIME,
        output_path=figure_path,
    )
    display(figure)
    print("Saved:", figure_path)

## Animations

The full executions are animated with `animate_dataset()`.

In [ ]:
for case_name, dataset_path in dataset_paths.items():
    animation_path = OUTPUT_DIRECTORY / f"{case_name}.gif"
    animate_dataset(
        dataset_path=dataset_path,
        config=visualization_config(case_name),
        output_path=animation_path,
        fps=6.0,
        dpi=100,
    )
    display(Image(filename=animation_path))
    print("Saved:", animation_path)

## Expected interpretation

Without strategic deconfliction, both aircraft reach the origin simultaneously, generating loss of separation and a collision event. With strategic deconfliction, UAV B receives a time-shifted counterproposal and crosses the same spatial intersection only after UAV A has cleared it.